In [ ]:
import glob
import os
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL"
BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"

FILE_PATTERN = "slp_{}_{}_run_*"

BATCH_SIZES = [64, 1024, 60000]

PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]


#=========================
#LOOP OVER BATCH SIZES
#=========================
for bs in BATCH_SIZES:
    all_avg_dfs = {}

    for p in PRUNING_PERCENTAGES:
        folder = os.path.join(BASE_DIR, BATCH_DIR_TEMPLATE.format(p, bs))
        pattern = FILE_PATTERN.format(p, bs)
        files = glob.glob(os.path.join(folder, pattern))
        # print(folder)
        # print(files)

        if not files:
            print(f"[WARNING] No files found for pruning {p} and batch size {bs}")
            continue

        # =========================
        # LOAD ALL RUNS
        # =========================
        dfs = []
        for f in files:
            df = pd.read_csv(f, sep=r"\s+")
            df.columns = df.columns.str.strip()

            df["CE_Train"] = pd.to_numeric(df["CE_Train"], errors="coerce")
            df["CE_TEST"] = pd.to_numeric(df["CE_TEST"], errors="coerce")
            df["Accuracy(%)"] = pd.to_numeric(df["Accuracy(%)"], errors="coerce")

            dfs.append(df)

        all_runs = pd.concat(dfs, ignore_index=True)

        # =========================
        # GROUP + AVERAGE
        # =========================
        avg_df = all_runs.groupby("Batch_Number", as_index=False).agg(
            Avg_CE_Train=("CE_Train", "mean"),
            Avg_CE_Test=("CE_TEST", "mean"),
            Avg_Accuracy=("Accuracy(%)", "mean"),
            Num_Runs=("CE_TEST", "count")
        )

        # =========================
        # SAVE CSV
        # =========================
        out_csv = os.path.join(folder, f"averaged_runs_p_{p}_bs_{bs}.csv")
        avg_df.to_csv(out_csv, index=False)

        all_avg_dfs[p] = avg_df

In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP-FMNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL"
BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"

BATCH_SIZES = [64, 1024, 60000]

PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# =========================
# OUTPUT DIRECTORIES
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

print(f"[INFO] Graphs → {AUC_GRAPH_DIR}")
print(f"[INFO] Data → {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# CE_MAX = maximum allowed cross-entropy value.
# np.log(10) is the CE of random guessing for a 10-class problem like FMNIST.
# So CE values above this get capped to avoid huge spikes distorting the graph.
CE_MAX = np.log(10)  # ≈ 2.3026

# =========================
# COLOR SCHEME
# =========================
# Explicit mapping so colors always match the pruning percentages,
# regardless of plotting order.

PRUNING_COLOR_MAP = {
    0.0:  "#17becf",  # 0%
    0.1:  "#B9D9EB",  # 10%
    0.2:  "#bcbd22",  # 20%
    0.3:  "#7f7f7f",  # 30%
    0.4:  "#e377c2",  # 40%
    0.5:  "#8c564b",  # 50%
    0.6:  "#800080",  # 60%
    0.7:  "#d62728",  # 70%
    0.8:  "#2ca02c",  # 80%
    0.9:   "#C5B0D5",
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]
COLOR_LIST = COLOR_LIST[::-1]


# =========================
# HELPER FOR CLEAN P% STRINGS
# =========================
def format_pruning_value(p):
    """
    Keeps values like:
    0.0 -> '0.0'
    0.1 -> '0.1'
    0.82 -> '0.82'
    1.0 -> '1.0'
    """
    s = f"{p:.2f}".rstrip('0')
    if s.endswith('.'):
        s += '0'
    return s

# =========================
# MAIN LOOP
# =========================
all_auc_records = []

for bs in BATCH_SIZES:

    # avg_dfs = dictionary of averaged dataframes.
    # Key = pruning percentage p
    # Value = the dataframe loaded from that pruning percentage's CSV file
    #
    # Example:
    # avg_dfs[0.9] = dataframe for 90% pruning
    # avg_dfs[0.0] = dataframe for 0% pruning
    avg_dfs = {}

    # -------------------------
    # LOAD AVERAGED FILES
    # -------------------------
    for p in PRUNING_PERCENTAGES:
        p_str = format_pruning_value(p)

        folder = os.path.join(BASE_DIR_ROOT, BATCH_DIR_TEMPLATE.format(p_str, bs))
        file_path = os.path.join(folder, f"averaged_runs_p_{p_str}_bs_{bs}.csv")

        if not os.path.isfile(file_path):
            print(f"[WARNING] File not found: {file_path}")
            continue

        df = pd.read_csv(file_path)

        # np.minimum(array, CE_MAX) compares each value in the array to CE_MAX
        # and keeps the smaller one.
        #
        # Example:
        # if Avg_CE_Train = [1.2, 2.0, 3.5]
        # and CE_MAX = 2.3026
        # result becomes [1.2, 2.0, 2.3026]
        #
        # So this caps all CE values so they cannot exceed CE_MAX.
        df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
        df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

        avg_dfs[p] = df

    if not avg_dfs:
        print(f"[WARNING] No data found for BS={bs}")
        continue

    # -------------------------
    # Find lowest max Batch_Number (exclude 100%)
    # -------------------------
    # non100_dfs = dictionary containing all pruning levels EXCEPT 1.0 (100% pruning)
    #
    # Why exclude 100% pruning?
    # Because 100% pruning is a special case: the model is fully pruned,
    # so it does not behave like the other curves.
    # Later in the code, 100% pruning is plotted as a flat horizontal line.
    non100_dfs = {p: df for p, df in avg_dfs.items() if p != 1.0}

    if non100_dfs:
        # min(...) here is the normal Python min function.
        # It looks at the maximum Batch_Number in each dataframe and chooses the smallest one.
        #
        # Example:
        # 0.0 pruning max batch = 500
        # 0.9 pruning max batch = 480
        # 0.98 pruning max batch = 430
        #
        # Then lowest_max_batch = 430
        #
        # This ensures all AUC calculations are compared over the same x-range.
        lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
    else:
        lowest_max_batch = min(df["Batch_Number"].max() for df in avg_dfs.values())

    # -------------------------
    # PLOT FUNCTION
    # -------------------------
    def plot_and_save_ce(avg_dfs, bs, ce_column, title):
        plt.figure(figsize=(10, 5))
        plt.title(f"SLP | FMNIST | BS={bs}")
        plt.xlabel("Batch Number")
        plt.ylabel(title)

        if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )
            
        for i, p in enumerate(sorted(avg_dfs.keys(), reverse=True)):
            df = avg_dfs[p]
            color = COLOR_LIST[i % len(COLOR_LIST)]

            if p == 1.0:
                ce_value = df[ce_column].iloc[-1]

                # min(value, CE_MAX) compares two single numbers
                # and keeps the smaller one.
                #
                # Example:
                # ce_value = 3.0
                # CE_MAX = 2.3026
                # result = 2.3026
                #
                # So this caps this single CE value for 100% pruning.
                ce_value = min(ce_value, CE_MAX)

                x = np.array([0, lowest_max_batch])
                y = np.array([ce_value, ce_value])
                auc = ce_value * lowest_max_batch

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.5,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )

            else:
                df_trunc = df[df["Batch_Number"] <= lowest_max_batch]
                x = df_trunc["Batch_Number"].to_numpy()

                # Again, np.minimum is being used element-by-element on the whole y array
                y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)

                mask = np.isfinite(x) & np.isfinite(y)
                x, y = x[mask], y[mask]

                if len(x) == 0:
                    print(f"[WARNING] NaNs only for P%={p}, BS={bs}")
                    continue

                auc = np.trapezoid(y, x)

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.0,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )
                plt.fill_between(x, y, alpha=0.2, color=color)

            all_auc_records.append([bs, p, ce_column, auc])

        plt.grid(True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
        plt.tight_layout()

        fig_path = os.path.join(AUC_GRAPH_DIR, f"{ce_column}_BS_{bs}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[SAVED] {fig_path}")

    # -------------------------
    # Plot Train & Test
    # -------------------------
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE Train")
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE Test")

# -------------------------
# SAVE SINGLE CSV FOR ALL
# -------------------------
if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\n✅ All AUC graphs and combined data saved successfully.")

[INFO] Graphs → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100
[INFO] Data → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_0-100
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Test_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Train_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-100\Avg_CE_Test_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_0-100\AUC_ALL_BS.csv

✅ All AUC graphs and combined data saved successfully.


In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP-FMNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL"
BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"

BATCH_SIZES = [64, 1024, 60000]

PRUNING_PERCENTAGES = [0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]

# =========================
# OUTPUT DIRECTORIES
# =======================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_80-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_80-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

print(f"[INFO] Graphs → {AUC_GRAPH_DIR}")
print(f"[INFO] Data → {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# Maximum CE cap
CE_MAX = np.log(10)  # ≈ 2.3026

# =========================
# COLOR SCHEME
# =========================
# Explicit mapping so colors always match the pruning percentages,
# regardless of plotting order.

PRUNING_COLOR_MAP = {
    0.8:  "#2ca02c",  # 80%
    0.82: "#FCCDE5",  # new
    0.84: "#FB8072",  # new
    0.86: "#A6D854",  # new
    0.88: "#FFFFB3",  # new
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",  # 90%
    0.94:  "#BEBADA",  # new
    0.96:  "#80B1D3",  # new
    0.96:  "#FDB462",  # new
    0.98:  "#C5B0D5",  # new
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]

# =========================
# HELPER FOR CLEAN P% STRINGS
# =========================
def format_pruning_value(p):
    """
    Keeps values like:
    0.0 -> '0.0'
    0.1 -> '0.1'
    0.82 -> '0.82'
    1.0 -> '1.0'
    """
    s = f"{p:.2f}".rstrip('0')
    if s.endswith('.'):
        s += '0'
    return s

# =========================
# MAIN LOOP
# =========================
all_auc_records = []

for bs in BATCH_SIZES:

    avg_dfs = {}

    # -------------------------
    # LOAD AVERAGED FILES
    # -------------------------
    for p in PRUNING_PERCENTAGES:
        p_str = format_pruning_value(p)

        folder = os.path.join(BASE_DIR_ROOT, BATCH_DIR_TEMPLATE.format(p_str, bs))
        file_path = os.path.join(folder, f"averaged_runs_p_{p_str}_bs_{bs}.csv")

        if not os.path.isfile(file_path):
            print(f"[WARNING] File not found: {file_path}")
            continue

        df = pd.read_csv(file_path)

        # Cap CE values
        df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
        df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

        avg_dfs[p] = df

    if not avg_dfs:
        print(f"[WARNING] No data found for BS={bs}")
        continue

    # -------------------------
    # Find lowest max Batch_Number (exclude 100%)
    # -------------------------
    non100_dfs = {p: df for p, df in avg_dfs.items() if p != 1.0}

    if non100_dfs:
        lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
    else:
        lowest_max_batch = min(df["Batch_Number"].max() for df in avg_dfs.values())

    # -------------------------
    # PLOT FUNCTION
    # -------------------------
    def plot_and_save_ce(avg_dfs, bs, ce_column, title):
        plt.figure(figsize=(10, 5))
        plt.title(f"SLP | FMNIST | BS={bs}")
        plt.xlabel("Batch Number")
        plt.ylabel(title)

        if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )

        # Plot in descending pruning order for legend display,
        # but colors are taken directly from the map.
        for p in sorted(avg_dfs.keys(), reverse=True):
            df = avg_dfs[p]
            color = PRUNING_COLOR_MAP.get(p, "#000000")

            if p == 1.0:
                ce_value = df[ce_column].iloc[-1]
                ce_value = min(ce_value, CE_MAX)

                x = np.array([0, lowest_max_batch])
                y = np.array([ce_value, ce_value])
                auc = ce_value * lowest_max_batch

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.5,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )

            else:
                df_trunc = df[df["Batch_Number"] <= lowest_max_batch]
                x = df_trunc["Batch_Number"].to_numpy()
                y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)

                mask = np.isfinite(x) & np.isfinite(y)
                x, y = x[mask], y[mask]

                if len(x) == 0:
                    print(f"[WARNING] NaNs only for P%={p}, BS={bs}")
                    continue

                auc = np.trapezoid(y, x)

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.0,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )
                plt.fill_between(x, y, alpha=0.2, color=color)

            all_auc_records.append([bs, p, ce_column, auc])

        plt.grid(True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
        plt.tight_layout()

        fig_path = os.path.join(AUC_GRAPH_DIR, f"{ce_column}_BS_{bs}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[SAVED] {fig_path}")

    # -------------------------
    # Plot Train & Test
    # -------------------------
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE Train")
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE Test")

# -------------------------
# SAVE SINGLE CSV FOR ALL
# -------------------------
if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\n✅ All AUC graphs and combined data saved successfully.")

[INFO] Graphs → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100
[INFO] Data → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_80-100
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Test_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Train_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_80-100\Avg_CE_Test_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_80-100\AUC_ALL_BS.csv

✅ All AUC graphs and combined data saved successfully.


In [22]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP-FMNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL"
BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"

BATCH_SIZES = [64, 1024, 60000]

PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]

# =========================
# OUTPUT DIRECTORIES
# =======================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-80-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-80-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

print(f"[INFO] Graphs → {AUC_GRAPH_DIR}")
print(f"[INFO] Data → {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# Maximum CE cap
CE_MAX = np.log(10)  # ≈ 2.3026

# =========================
# COLOR SCHEME
# =========================
# Explicit mapping so colors always match the pruning percentages,
# regardless of plotting order.

PRUNING_COLOR_MAP = {
    0.0:  "#17becf",  # 0%
    0.1:  "#B9D9EB",  # 10%
    0.2:  "#bcbd22",  # 20%
    0.3:  "#7f7f7f",  # 30%
    0.4:  "#e377c2",  # 40%
    0.5:  "#8c564b",  # 50%
    0.6:  "#800080",  # 60%
    0.7:  "#d62728",  # 70%
    0.8:  "#2ca02c",  # 80%
    0.82: "#FCCDE5",  # new
    0.84: "#FB8072",  # new
    0.86: "#A6D854",  # new
    0.88: "#FFFFB3",  # new
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",  # 90%
    0.94:  "#BEBADA",  # new
    0.96:  "#80B1D3",  # new
    0.96:  "#FDB462",  # new
    0.98:  "#C5B0D5",  # new
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]

# =========================
# HELPER FOR CLEAN P% STRINGS
# =========================
def format_pruning_value(p):
    """
    Keeps values like:
    0.0 -> '0.0'
    0.1 -> '0.1'
    0.82 -> '0.82'
    1.0 -> '1.0'
    """
    s = f"{p:.2f}".rstrip('0')
    if s.endswith('.'):
        s += '0'
    return s

# =========================
# MAIN LOOP
# =========================
all_auc_records = []

for bs in BATCH_SIZES:

    avg_dfs = {}

    # -------------------------
    # LOAD AVERAGED FILES
    # -------------------------
    for p in PRUNING_PERCENTAGES:
        p_str = format_pruning_value(p)

        folder = os.path.join(BASE_DIR_ROOT, BATCH_DIR_TEMPLATE.format(p_str, bs))
        file_path = os.path.join(folder, f"averaged_runs_p_{p_str}_bs_{bs}.csv")

        if not os.path.isfile(file_path):
            print(f"[WARNING] File not found: {file_path}")
            continue

        df = pd.read_csv(file_path)

        # Cap CE values
        df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
        df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

        avg_dfs[p] = df

    if not avg_dfs:
        print(f"[WARNING] No data found for BS={bs}")
        continue

    # -------------------------
    # Find lowest max Batch_Number (exclude 100%)
    # -------------------------
    non100_dfs = {p: df for p, df in avg_dfs.items() if p != 1.0}

    if non100_dfs:
        lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
    else:
        lowest_max_batch = min(df["Batch_Number"].max() for df in avg_dfs.values())

    # -------------------------
    # PLOT FUNCTION
    # -------------------------
    def plot_and_save_ce(avg_dfs, bs, ce_column, title):
        plt.figure(figsize=(10, 5))
        plt.title(f"SLP | FMNIST | BS={bs}")
        plt.xlabel("Batch Number")
        plt.ylabel(title)

        if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )

        # Plot in descending pruning order for legend display,
        # but colors are taken directly from the map.
        for p in sorted(avg_dfs.keys(), reverse=True):
            df = avg_dfs[p]
            color = PRUNING_COLOR_MAP.get(p, "#000000")

            if p == 1.0:
                ce_value = df[ce_column].iloc[-1]
                ce_value = min(ce_value, CE_MAX)

                x = np.array([0, lowest_max_batch])
                y = np.array([ce_value, ce_value])
                auc = ce_value * lowest_max_batch

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.5,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )

            else:
                df_trunc = df[df["Batch_Number"] <= lowest_max_batch]
                x = df_trunc["Batch_Number"].to_numpy()
                y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)

                mask = np.isfinite(x) & np.isfinite(y)
                x, y = x[mask], y[mask]

                if len(x) == 0:
                    print(f"[WARNING] NaNs only for P%={p}, BS={bs}")
                    continue

                auc = np.trapezoid(y, x)

                plt.plot(
                    x, y,
                    color=color,
                    linewidth=2.0,
                    label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                )
                plt.fill_between(x, y, alpha=0.2, color=color)

            all_auc_records.append([bs, p, ce_column, auc])

        plt.grid(True)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
        plt.tight_layout()

        fig_path = os.path.join(AUC_GRAPH_DIR, f"{ce_column}_BS_{bs}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[SAVED] {fig_path}")

    # -------------------------
    # Plot Train & Test
    # -------------------------
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE Train")
    plot_and_save_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE Test")

# -------------------------
# SAVE SINGLE CSV FOR ALL
# -------------------------
if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\n✅ All AUC graphs and combined data saved successfully.")

[INFO] Graphs → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100
[INFO] Data → C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_0-80-100
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Test_BS_1024.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Train_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_graph_0-80-100\Avg_CE_Test_BS_60000.png
[SAVED] C:\Users\Micah\physlab\SLP-FMNIST\prune_layers_ALL\AUC_data_0-80-100\AUC_ALL_BS.csv

✅ All AUC graphs and combined data saved successfully.
